# Task 1 — Notebook 01: Data Ingestion (CDL + NDVI)

**Purpose (NAFSI Task 1):** Load **CDL** as a crop mask and **MODIS NDVI** time series for the Corn Belt (Iowa / Nebraska focus in config), compare corn vs. soybean phenology. Data are read from **interim NetCDF** produced by `scripts/build_interim_data.py` (already EPSG:5070).

**Inputs (interim):**  
- `data/interim/cdl/cdl_stack_{y0}_{y1}.nc` — CDL years in one stack (`year`, `y`, `x`); notebook slices the config year  
- `data/interim/ndvi/ndvi_weekly_{year}.nc` — weekly NDVI for that calendar year (`time`, `y`, `x`)

**Processed (tabular preview):** `data/processed/cdl/cdl_stack_wide.parquet`, `data/processed/ndvi/ndvi_weekly_{year}_wide.parquet`, plus JSON sidecars — column catalog and sample rows at the end.

**This notebook:** loads CDL stack + NDVI for the analysis year from `configs/task1_ndvi_analysis.yaml`. QA, smoothing, and exports are in later notebooks.

In [1]:
# NOTE: This notebook was scaffolded with AI assistance. All analysis logic
# and interpretation must be reviewed and authored by the team.

import json
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import xarray as xr
import yaml
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 48)

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (requirements.txt + src/). cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.interim_loaders import load_cdl_stack_from_interim

with open(REPO_ROOT / "configs" / "task1_ndvi_analysis.yaml") as f:
    cfg = yaml.safe_load(f)

cdl_year = int(cfg["cdl"]["year"])
ndvi_year = int(cfg["ndvi"]["year"])
print("Config:", "CDL year", cdl_year, "| NDVI year", ndvi_year)

Config: CDL year 2022 | NDVI year 2022


In [2]:
# Full CDL stack from interim; slice analysis year for crop mask (corn / soybean).
cdl_stack = load_cdl_stack_from_interim(REPO_ROOT)
_years = {int(y) for y in cdl_stack["year"].values}
if cdl_year not in _years:
    raise ValueError(f"Config CDL year {cdl_year} not in interim stack years {sorted(_years)}")
cdl = cdl_stack.sel(year=cdl_year)
print("cdl_stack", dict(cdl_stack.sizes), "| cdl (mask year)", dict(cdl.sizes), cdl.dtype)


cdl_stack {'year': 18, 'y': 1520, 'x': 2048} | cdl (mask year) {'y': 1520, 'x': 2048} int16


In [3]:
# NDVI for the **analysis year only** (one interim file). Same grid as the pipeline;
# use ``load_ndvi_weekly_all_years(REPO_ROOT)`` if you need every ``calendar_year`` stacked.
from src.io.interim_loaders import _open_interim_nc

ndvi_path = REPO_ROOT / "data" / "interim" / "ndvi" / f"ndvi_weekly_{ndvi_year}.nc"
if not ndvi_path.is_file():
    raise FileNotFoundError(
        f"Missing {ndvi_path}. Run: python scripts/build_interim_data.py --dataset ndvi"
    )
_da_year = _open_interim_nc(ndvi_path)["ndvi"]
ndvi = _da_year.rename("ndvi")
ndvi_all = ndvi.expand_dims(calendar_year=[ndvi_year])
print("ndvi (analysis year)", dict(ndvi.sizes), "| file:", ndvi_path.name)


ndvi (analysis year) {'time': 23, 'y': 1520, 'x': 2048} | file: ndvi_weekly_2022.nc


In [4]:
# Interim stacks match the download grid (CONUS Albers EPSG:5070 in the pipeline).
# Next notebooks: align CDL → NDVI resolution if needed, purity filter, QA, smoothing.

try:
    import rioxarray  # noqa: F401

    print("CDL CRS:", getattr(cdl.rio, "crs", None))
    print("NDVI CRS:", getattr(ndvi.rio, "crs", None))
except Exception as exc:
    print("CRS check skipped (optional rioxarray .rio):", exc)


CRS check skipped (optional rioxarray .rio): No module named 'rioxarray'


## Raster stacks (interim NetCDF)

`cdl_stack` / `ndvi_all` are **gridded** `xarray.DataArray` objects. Below: dimension summary, then **processed Parquet** (one row per pixel `iy`, `ix`) with **every column name**, dtypes, JSON sidecars, and a short sample table.

In [6]:
def _da_profile(name: str, da: xr.DataArray) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"array": name, "field": "dims (sizes)", "value": str(dict(da.sizes))},
            {"array": name, "field": "dtype", "value": str(da.dtype)},
            {"array": name, "field": "coords", "value": ", ".join(map(str, da.coords))},
            {"array": name, "field": "attrs (keys)", "value": ", ".join(sorted(map(str, da.attrs.keys()))) or "—"},
        ]
    )


def _parquet_columns(path: Path) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    sch = pf.schema_arrow
    rows = [{"column": name, "pyarrow_type": str(sch.field(name).type)} for name in sch.names]
    out = pd.DataFrame(rows)
    out.insert(0, "parquet_file", path.name)
    return out


def _parquet_head(path: Path, n: int = 8) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    batch = next(pf.iter_batches(batch_size=n))
    return batch.to_pandas()


print("— Interim stacks (gridded) —")
display(pd.concat([_da_profile("cdl_stack", cdl_stack), _da_profile("cdl (mask year)", cdl)], ignore_index=True))
display(pd.concat([_da_profile("ndvi_all", ndvi_all), _da_profile("ndvi (analysis year)", ndvi)], ignore_index=True))

# Processed wide tables: full column list (schema) + sample rows + JSON sidecars
cdl_pq = REPO_ROOT / "data" / "processed" / "cdl" / "cdl_stack_wide.parquet"
ndvi_pq = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{ndvi_year}_wide.parquet"
cdl_meta = REPO_ROOT / "data" / "processed" / "cdl" / "cdl_stack_spatial_metadata.json"
ndvi_meta = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{ndvi_year}_metadata.json"

print("\n— CDL processed Parquet (all columns) —")
if cdl_pq.is_file():
    pf = pq.ParquetFile(cdl_pq)
    print(f"rows (approx): {pf.metadata.num_rows:,} | row_groups: {pf.num_row_groups}")
    display(_parquet_columns(cdl_pq))
    display(_parquet_head(cdl_pq, n=8))
else:
    print(f"Missing {cdl_pq} — run: python scripts/process_interim_to_parquet.py --dataset cdl")

if cdl_meta.is_file():
    print("\nCDL spatial metadata JSON (flattened)")
    display(pd.json_normalize(json.loads(cdl_meta.read_text(encoding='utf-8')), sep='_'))

print(f"\n— NDVI processed Parquet ({ndvi_year}) — all columns —")
if ndvi_pq.is_file():
    pf = pq.ParquetFile(ndvi_pq)
    print(f"rows (approx): {pf.metadata.num_rows:,} | row_groups: {pf.num_row_groups}")
    display(_parquet_columns(ndvi_pq))
    display(_parquet_head(ndvi_pq, n=8))
else:
    print(f"Missing {ndvi_pq} — run: python scripts/process_interim_to_parquet.py --dataset ndvi --year {ndvi_year}")

if ndvi_meta.is_file():
    print("\nNDVI metadata JSON (flattened)")
    meta = json.loads(ndvi_meta.read_text(encoding="utf-8"))
    # time_start_day can be long — show first/last few entries in a small table if list-like
    flat = {k: v for k, v in meta.items() if k != "time_start_day"}
    display(pd.json_normalize([flat], sep="_"))
    tsd = meta.get("time_start_day")
    if isinstance(tsd, list) and len(tsd) > 0:
        tdf = pd.DataFrame({"week_index": range(len(tsd)), "time_start_day": tsd})
        print("Per-week column mapping: w000 … ↔ time_start_day (head / tail)")
        display(pd.concat([tdf.head(6), tdf.tail(6)], ignore_index=True))

— Interim stacks (gridded) —


,array,field,value
0,cdl_stack,dims (sizes),"{'year': 18, 'y': 1520, 'x': 2048}"
1,cdl_stack,dtype,int16
2,cdl_stack,coords,"year, x, y, spatial_ref"
3,cdl_stack,attrs (keys),"AREA_OR_POINT, TIFFTAG_RESOLUTIONUNIT, TIFFT..."
4,cdl (mask year),dims (sizes),"{'y': 1520, 'x': 2048}"
5,cdl (mask year),dtype,int16
6,cdl (mask year),coords,"year, x, y, spatial_ref"
7,cdl (mask year),attrs (keys),"AREA_OR_POINT, TIFFTAG_RESOLUTIONUNIT, TIFFT..."


,array,field,value
0,ndvi_all,dims (sizes),"{'calendar_year': 1, 'time': 23, 'y': 1520, ..."
1,ndvi_all,dtype,float64
2,ndvi_all,coords,"calendar_year, time, x, y"
3,ndvi_all,attrs (keys),"AREA_OR_POINT, TIFFTAG_RESOLUTIONUNIT, TIFFT..."
4,ndvi (analysis year),dims (sizes),"{'time': 23, 'y': 1520, 'x': 2048}"
5,ndvi (analysis year),dtype,float64
6,ndvi (analysis year),coords,"time, x, y"
7,ndvi (analysis year),attrs (keys),"AREA_OR_POINT, TIFFTAG_RESOLUTIONUNIT, TIFFT..."



— CDL processed Parquet (all columns) —
rows (approx): 3,112,960 | row_groups: 3


,parquet_file,column,pyarrow_type
0,cdl_stack_wide.parquet,iy,int32
1,cdl_stack_wide.parquet,ix,int32
2,cdl_stack_wide.parquet,cdl_2008,int32
3,cdl_stack_wide.parquet,cdl_2009,int32
4,cdl_stack_wide.parquet,cdl_2010,int32
5,cdl_stack_wide.parquet,cdl_2011,int32
6,cdl_stack_wide.parquet,cdl_2012,int32
7,cdl_stack_wide.parquet,cdl_2013,int32
8,cdl_stack_wide.parquet,cdl_2014,int32
9,cdl_stack_wide.parquet,cdl_2015,int32


...

,iy,ix,w000,w001,w002,w003,w004,w005,w006,w007,w008,w009,w010,w011,w012,w013,w014,w015,w016,w017,w018,w019,w020,w021,w022
0,0,0,178.0,176.0,175.0,175.0,183.0,184.0,212.0,180.0,175.0,184.0,159.0,159.0,174.0,160.0,163.0,158.0,155.0,157.0,162.0,160.0,166.0,175.0,178.0
1,0,1,178.0,171.0,178.0,173.0,183.0,179.0,212.0,180.0,170.0,174.0,159.0,157.0,173.0,160.0,160.0,156.0,154.0,156.0,162.0,157.0,161.0,175.0,178.0
2,0,2,175.0,171.0,170.0,170.0,178.0,174.0,208.0,179.0,170.0,169.0,161.0,157.0,166.0,158.0,160.0,156.0,155.0,153.0,162.0,155.0,158.0,163.0,165.0
3,0,3,173.0,172.0,171.0,171.0,181.0,176.0,211.0,174.0,169.0,166.0,162.0,156.0,167.0,158.0,161.0,153.0,153.0,154.0,159.0,156.0,160.0,161.0,222.0
4,0,4,172.0,173.0,169.0,170.0,178.0,173.0,207.0,174.0,167.0,166.0,160.0,159.0,166.0,157.0,157.0,154.0,153.0,154.0,159.0,156.0,162.0,164.0,194.0
5,0,5,171.0,173.0,169.0,172.0,178.0,178.0,207.0,177.0,170.0,166.0,160.0,159.0,166.0,156.0,157.0,155.0,154.0,154.0,160.0,160.0,162.0,164.0,209.0
6,0,6,170.0,175.0,172.0,173.0,182.0,182.0,212.0,183.0,175.0,170.0,165.0,163.0,165.0,157.0,157.0,155.0,154.0,154.0,159.0,159.0,160.0,166.0,171.0
7,0,7,171.0,176.0,174.0,174.0,193.0,181.0,214.0,182.0,175.0,172.0,168.0,163.0,164.0,156.0,157.0,155.0,154.0,154.0,159.0,157.0,160.0,166.0,172.0



NDVI metadata JSON (flattened)


,source_nc,year,height,width,n_time,wide_columns,note,crs,transform
0,data\interim\ndvi\ndvi_weekly_2022.nc,2022,1520,2048,23,"[iy, ix, w000, w001, w002, w003, w004, w005,...",w000.. are weekly NDVI layers in the same or...,None,"[320.3125, 0.0, -843000.0, 0.0, -319.0789473..."


Per-week column mapping: w000 … ↔ time_start_day (head / tail)


,week_index,time_start_day
0,0,2022-05-02
1,1,2022-05-09
2,2,2022-05-16
3,3,2022-05-23
4,4,2022-05-30
5,5,2022-06-06
6,17,2022-08-29
7,18,2022-09-05
8,19,2022-09-12
9,20,2022-09-19
